# Optimasi SVR Menggunakan PSO untuk Peramalan Inflasi Indonesia

**Final Project – Rekayasa Komputasional**

Notebook ini menggabungkan seluruh tahapan pipeline:
1. Setup & Instalasi
2. Preprocessing Data
3. Baseline SVR
4. Optimasi PSO
5. Evaluasi & Visualisasi


## 1. Setup & Instalasi Dependensi

Jalankan sel berikut untuk menginstal semua library yang diperlukan.

In [ ]:
# Instal semua dependensi yang diperlukan
!pip install numpy pandas scikit-learn matplotlib pyswarms openpyxl -q

In [ ]:
# Import library standar
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Import sklearn
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

# Import pyswarms
from pyswarms.single import GlobalBestPSO

# Set seed untuk reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Setup selesai. Semua library berhasil diimpor.')

## 2. Upload & Load Data

Upload file `inflasi.csv` dari folder `data/`, atau gunakan data dummy di bawah.

In [ ]:
# Opsi A: Upload file dari lokal (uncomment jika di Google Colab)
# from google.colab import files
# uploaded = files.upload()  # Upload inflasi.csv
# PATH_CSV = 'inflasi.csv'

# Opsi B: Gunakan path relatif (jika menjalankan dari root project)
PATH_CSV = os.path.join('data', 'inflasi.csv')

# Baca data
df = pd.read_csv(PATH_CSV)
df['periode'] = pd.to_datetime(df['periode'])
df = df.sort_values('periode').reset_index(drop=True)

print(f'Data berhasil dimuat: {len(df)} baris')
print(f'Rentang waktu: {df["periode"].min().strftime("%b %Y")} s.d. {df["periode"].max().strftime("%b %Y")}')
df.head()

## 3. Eksplorasi Data

In [ ]:
# Statistik deskriptif
print('Statistik Deskriptif:')
print(df['inflasi'].describe())

# Visualisasi data inflasi
plt.figure(figsize=(14, 5))
plt.plot(df['periode'], df['inflasi'], color='steelblue', linewidth=1.5)
plt.title('Data Inflasi Bulanan Indonesia (2010–2026)', fontsize=14)
plt.xlabel('Periode')
plt.ylabel('Inflasi (%)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Preprocessing Data

- Buat fitur lag (sliding window 12 lag)
- Normalisasi dengan MinMaxScaler
- Split kronologis 80:20

In [ ]:
JUMLAH_LAG = 12  # Jumlah lag features

# Fungsi untuk membuat fitur lag
def buat_fitur_lag(series, jumlah_lag):
    """Membuat matriks fitur lag dari time series."""
    X, y = [], []
    for i in range(jumlah_lag, len(series)):
        X.append(series[i - jumlah_lag:i])
        y.append(series[i])
    return np.array(X), np.array(y)

# Normalisasi data
nilai_inflasi = df['inflasi'].values.astype(float)
scaler = MinMaxScaler(feature_range=(0, 1))
nilai_dinorm = scaler.fit_transform(nilai_inflasi.reshape(-1, 1)).flatten()

# Buat lag features
X, y = buat_fitur_lag(nilai_dinorm, JUMLAH_LAG)

# Split kronologis 80:20
n_train = int(len(X) * 0.8)
X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]

print(f'Total sampel      : {len(X)}')
print(f'Data train        : {X_train.shape[0]} sampel')
print(f'Data test         : {X_test.shape[0]} sampel')
print(f'Jumlah fitur (lag): {X_train.shape[1]}')

## 5. Baseline SVR (Hyperparameter Default)

In [ ]:
# Fungsi menghitung MAPE
def hitung_mape(y_aktual, y_prediksi):
    mask = y_aktual != 0
    return np.mean(np.abs((y_aktual[mask] - y_prediksi[mask]) / y_aktual[mask])) * 100

# Latih SVR dengan hyperparameter default
model_baseline = SVR(kernel='rbf')  # C=1.0, gamma='scale', epsilon=0.1
model_baseline.fit(X_train, y_train)

# Prediksi & inverse transform
y_pred_b_norm = model_baseline.predict(X_test)
y_test_asli   = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_b_asli = scaler.inverse_transform(y_pred_b_norm.reshape(-1, 1)).flatten()

# Hitung metrik
rmse_b = np.sqrt(mean_squared_error(y_test_asli, y_pred_b_asli))
mae_b  = mean_absolute_error(y_test_asli, y_pred_b_asli)
mape_b = hitung_mape(y_test_asli, y_pred_b_asli)

print('=== Baseline SVR ===')
print(f'  RMSE : {rmse_b:.4f}')
print(f'  MAE  : {mae_b:.4f}')
print(f'  MAPE : {mape_b:.4f}%')

## 6. Optimasi Hyperparameter SVR dengan PSO

- 20 partikel, 30 iterasi
- c1 = c2 = 1.5, w = 0.7
- Fungsi fitness: rata-rata RMSE TimeSeriesSplit 3-fold

In [ ]:
# Konfigurasi PSO
JUMLAH_PARTIKEL = 20
JUMLAH_ITERASI  = 30
JUMLAH_FOLD     = 3

def fungsi_fitness(params, X_train, y_train):
    """Fungsi fitness PSO: rata-rata RMSE TimeSeriesSplit."""
    n_partikel     = params.shape[0]
    fitness_values = np.zeros(n_partikel)
    tscv           = TimeSeriesSplit(n_splits=JUMLAH_FOLD)

    for i in range(n_partikel):
        C_val, gamma_val, eps_val = params[i, 0], params[i, 1], params[i, 2]
        rmse_folds = []
        for idx_tr, idx_val in tscv.split(X_train):
            X_tr, X_vl = X_train[idx_tr], X_train[idx_val]
            y_tr, y_vl = y_train[idx_tr], y_train[idx_val]
            svr = SVR(kernel='rbf', C=C_val, gamma=gamma_val, epsilon=eps_val)
            svr.fit(X_tr, y_tr)
            y_p = svr.predict(X_vl)
            rmse_folds.append(np.sqrt(mean_squared_error(y_vl, y_p)))
        fitness_values[i] = np.mean(rmse_folds)
    return fitness_values

# Konfigurasi PSO
opsi_pso = {'c1': 1.5, 'c2': 1.5, 'w': 0.7}
bounds   = (np.array([0.1, 0.0001, 0.0001]),
            np.array([1000.0, 10.0, 0.5]))

optimizer = GlobalBestPSO(
    n_particles=JUMLAH_PARTIKEL,
    dimensions=3,
    options=opsi_pso,
    bounds=bounds,
)

print('Menjalankan PSO...')
best_cost, best_pos = optimizer.optimize(
    fungsi_fitness,
    iters=JUMLAH_ITERASI,
    X_train=X_train,
    y_train=y_train,
    verbose=True,
)

cost_history = optimizer.cost_history

print(f'\nHyperparameter terbaik:')
print(f'  C       = {best_pos[0]:.4f}')
print(f'  gamma   = {best_pos[1]:.6f}')
print(f'  epsilon = {best_pos[2]:.6f}')
print(f'  RMSE CV = {best_cost:.6f}')

In [ ]:
# Visualisasi kurva konvergensi PSO
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(cost_history) + 1), cost_history,
         color='steelblue', linewidth=2, marker='o', markersize=4)
plt.title('Kurva Konvergensi PSO\n(Optimasi Hyperparameter SVR)', fontsize=14)
plt.xlabel('Iterasi')
plt.ylabel('Nilai Fitness Terbaik (RMSE CV)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 7. Latih SVR dengan Hyperparameter Terbaik (SVR-PSO)

In [ ]:
# Latih ulang SVR dengan hyperparameter terbaik
model_pso = SVR(
    kernel='rbf',
    C=best_pos[0],
    gamma=best_pos[1],
    epsilon=best_pos[2]
)
model_pso.fit(X_train, y_train)

# Prediksi & inverse transform
y_pred_p_norm = model_pso.predict(X_test)
y_pred_p_asli = scaler.inverse_transform(y_pred_p_norm.reshape(-1, 1)).flatten()

# Hitung metrik
rmse_p = np.sqrt(mean_squared_error(y_test_asli, y_pred_p_asli))
mae_p  = mean_absolute_error(y_test_asli, y_pred_p_asli)
mape_p = hitung_mape(y_test_asli, y_pred_p_asli)

print('=== SVR-PSO ===')
print(f'  RMSE : {rmse_p:.4f}')
print(f'  MAE  : {mae_p:.4f}')
print(f'  MAPE : {mape_p:.4f}%')

## 8. Evaluasi & Perbandingan Model

In [ ]:
# Fungsi menghitung persen perbaikan
def persen_perbaikan(baseline, baru):
    return ((baseline - baru) / baseline) * 100 if baseline != 0 else 0.0

# Tabel perbandingan
tabel = pd.DataFrame({
    'Metrik'        : ['RMSE', 'MAE', 'MAPE (%)'],
    'Baseline SVR'  : [round(rmse_b, 4), round(mae_b, 4), round(mape_b, 4)],
    'SVR-PSO'       : [round(rmse_p, 4), round(mae_p, 4), round(mape_p, 4)],
    'Perbaikan (%)' : [
        round(persen_perbaikan(rmse_b, rmse_p), 2),
        round(persen_perbaikan(mae_b, mae_p), 2),
        round(persen_perbaikan(mape_b, mape_p), 2),
    ]
})

print('=== Tabel Perbandingan Metrik ===')
print(tabel.to_string(index=False))

# Simpan ke CSV
os.makedirs('hasil', exist_ok=True)
tabel.to_csv('hasil/perbandingan.csv', index=False)
print('\nTabel disimpan ke hasil/perbandingan.csv')

In [ ]:
# Grafik batang perbandingan metrik
metrik_list    = tabel['Metrik'].tolist()
nilai_baseline = tabel['Baseline SVR'].tolist()
nilai_pso_list = tabel['SVR-PSO'].tolist()

x = np.arange(len(metrik_list))
lebar = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
batang1 = ax.bar(x - lebar/2, nilai_baseline, lebar,
                 label='Baseline SVR', color='steelblue', alpha=0.8, edgecolor='black')
batang2 = ax.bar(x + lebar/2, nilai_pso_list, lebar,
                 label='SVR-PSO', color='darkorange', alpha=0.8, edgecolor='black')

for batang in [batang1, batang2]:
    for rect in batang:
        h = rect.get_height()
        ax.annotate(f'{h:.4f}',
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9)

ax.set_title('Perbandingan Metrik: Baseline SVR vs SVR-PSO', fontsize=14)
ax.set_xlabel('Metrik')
ax.set_ylabel('Nilai Metrik')
ax.set_xticks(x)
ax.set_xticklabels(metrik_list)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
os.makedirs('hasil/grafik', exist_ok=True)
plt.savefig('hasil/grafik/perbandingan_metrik.png', dpi=150)
plt.show()

In [ ]:
# Grafik prediksi vs aktual pada test set
n_total  = len(df) - JUMLAH_LAG
idx_test = range(n_train, n_total)
periode_test = df['periode'].iloc[
    [i + JUMLAH_LAG for i in idx_test]
].dt.strftime('%Y-%m').tolist()

plt.figure(figsize=(14, 6))
plt.plot(periode_test, y_test_asli,   label='Aktual',       color='black',
         linewidth=2, marker='o', markersize=4)
plt.plot(periode_test, y_pred_b_asli, label='Baseline SVR', color='steelblue',
         linewidth=1.5, linestyle='--')
plt.plot(periode_test, y_pred_p_asli, label='SVR-PSO',      color='darkorange',
         linewidth=1.5, linestyle='-.')

plt.title('Perbandingan Prediksi Inflasi Indonesia (Test Set)', fontsize=14)
plt.xlabel('Periode')
plt.ylabel('Inflasi (%)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('hasil/grafik/prediksi_test.png', dpi=150)
plt.show()

## 9. Kesimpulan

Notebook ini telah menjalankan seluruh pipeline:

1. **Preprocessing**: Lag features 12 bulan, MinMaxScaler, split 80:20 kronologis.
2. **Baseline SVR**: Evaluasi dengan RMSE, MAE, MAPE menggunakan hyperparameter default.
3. **PSO Optimization**: 20 partikel × 30 iterasi mencari C, gamma, epsilon optimal.
4. **SVR-PSO**: Pelatihan ulang dengan hyperparameter terbaik dan evaluasi pada test set.
5. **Perbandingan**: Tabel dan grafik menunjukkan peningkatan performa SVR-PSO dibandingkan baseline.

File output tersimpan di folder `hasil/`.